In [1]:
import polars as pl

# Filter ratings to at least 100 votes

In [2]:
df_ratings = pl.read_csv("../data/raw/title.ratings.tsv",separator="\t")
df_ratings = df_ratings.filter(pl.col("numVotes") >= 100_000)

print("len",len(df_ratings))
df_ratings.head()

len 3127


tconst,averageRating,numVotes
str,f64,i64
"""tt0012349""",8.2,146444
"""tt0013442""",7.8,123931
"""tt0015864""",8.1,128674
"""tt0017136""",8.2,202259
"""tt0017925""",8.1,105605


# Filter related metadata

In [3]:
df_titles = pl.scan_csv("../data/raw/title.basics.tsv",separator="\t", null_values=["\\N"],quote_char=None)

df_titles = (
    df_titles
    .filter(pl.col("tconst").is_not_null())
    .filter(pl.col("startYear").is_not_null())
    .filter(pl.col("runtimeMinutes").is_not_null())
    .filter(pl.col("primaryTitle").is_not_null())
    .filter(pl.col("originalTitle").is_not_null())
    
    .filter(pl.col("tconst").is_in(df_ratings["tconst"]))
    .filter(pl.col("titleType") == "movie")
    .filter(pl.col("runtimeMinutes") >= 60)
    .filter(pl.col("startYear") <= 2018)
    .filter(pl.col("isAdult") == 0)
    .collect()
)

print("len",len(df_titles))
df_titles.head()

/tmp/ipykernel_1053529/3219488026.py:16: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .collect()


len 2313


tconst,titleType,primaryTitle,originalTitle,isAdult,startYear,endYear,runtimeMinutes,genres
str,str,str,str,i64,i64,str,i64,str
"""tt0012349""","""movie""","""The Kid""","""The Kid""",0,1921,null,68,"""Comedy,Drama,Family"""
"""tt0013442""","""movie""","""Nosferatu""","""Nosferatu, eine Symphonie des …",0,1922,null,95,"""Fantasy,Horror"""
"""tt0015864""","""movie""","""The Gold Rush""","""The Gold Rush""",0,1925,null,95,"""Adventure,Comedy,Drama"""
"""tt0017136""","""movie""","""Metropolis""","""Metropolis""",0,1927,null,150,"""Drama,Sci-Fi"""
"""tt0017925""","""movie""","""The General""","""The General""",0,1926,null,78,"""Action,Adventure,Comedy"""


# Merge all metadata and store

In [4]:
merged_raw = df_titles.join(df_ratings,on="tconst",how="inner", suffix="_ratings").sort(pl.col("numVotes"))

merged_trim = (
    merged_raw
    .with_columns(
        pl.col("tconst").alias("imdb_id"),

    ).select(
        pl.col("imdb_id", "averageRating","numVotes","primaryTitle", "originalTitle", "runtimeMinutes", "genres" ,"titleType", "startYear")
    )
)

print("len",len(merged_trim))
merged_trim.sample(10)

len 2313


imdb_id,averageRating,numVotes,primaryTitle,originalTitle,runtimeMinutes,genres,titleType,startYear
str,f64,i64,str,str,i64,str,str,i64
"""tt0060196""",8.8,895686,"""The Good, the Bad and the Ugly""","""Il buono, il brutto, il cattiv…",178,"""Adventure,Drama,Western""","""movie""",1966
"""tt0055630""",8.2,144000,"""Yojimbo""","""Yôjinbô""",110,"""Action,Drama,Thriller""","""movie""",1961
"""tt0091064""",7.6,226107,"""The Fly""","""The Fly""",96,"""Drama,Horror,Sci-Fi""","""movie""",1986
"""tt2798920""",6.8,396612,"""Annihilation""","""Annihilation""",115,"""Adventure,Drama,Horror""","""movie""",2018
"""tt0393162""",7.4,194257,"""Coach Carter""","""Coach Carter""",136,"""Biography,Drama,Sport""","""movie""",2005
"""tt0404390""",7.3,107189,"""Running Scared""","""Running Scared""",122,"""Action,Crime,Drama""","""movie""",2006
"""tt0330793""",6.4,181981,"""The Punisher""","""The Punisher""",124,"""Action,Crime,Drama""","""movie""",2004
"""tt0898367""",7.2,272450,"""The Road""","""The Road""",111,"""Drama,Thriller""","""movie""",2009
"""tt0132477""",7.8,112616,"""October Sky""","""October Sky""",108,"""Biography,Drama,Family""","""movie""",1999


# Store data

In [5]:
from plot_shift import PlotShiftSqliteDB,MovieMeta

# merged_raw.write_csv(
#     "../data/proceeded/merged_raw.csv", separator="\t", include_header=True
# )

# merged_trim.write_csv(
#     "../data/proceeded/merged.csv", separator="\t", include_header=True
# )

# merged_trim.select("imdb_id").write_csv(
#     "../data/proceeded/imdb_ids.txt", include_header=False
# )

# movies_list = list(merged_trim.iter_rows())
movies_list = []
for idx, row in enumerate(merged_trim.iter_rows(), start=1):
    movie = MovieMeta(
        imdb_id=row[0],
        average_rating=row[1],
        num_votes=row[2],
        primary_title=row[3],
        original_title=row[4],
        runtime_minutes=row[5],
        genres=row[6],
        title_type=row[7],
        start_year=row[8],
    )
    movies_list.append(movie)


repo = PlotShiftSqliteDB("../data/proceeded/data.db")
repo.movies_meta.fresh_table()
repo.movies_meta.insert_many(movies_list)